# Phase 2: RAG Query Only (Load Existing Vector DB)

This notebook:
1. Loads the BGE-large embedding model
2. Connects to existing Milvus vector database (`milvus_all_docs_mixed.db`)
3. Verifies collection exists and shows statistics
4. Initializes Gemini 2.5 Flash for answer generation
5. Provides RAG query interface for question answering

**Use this notebook when:**
- Your vector database is already created and populated
- You just want to perform RAG queries without re-processing documents
- You want fast startup without chunking/embedding overhead

**IMPORTANT - Before running this notebook:**
- This assumes you've already run `phase2_chunking_embedding_mixed.ipynb` to create the vector database
- ⚠️ **Database locking issue**: Milvus Lite (SQLite-based) only allows ONE connection at a time
- **If you see "zero vectors" or connection errors:**
  1. Shut down the kernel for `phase2_chunking_embedding_mixed.ipynb` (Kernel → Shut Down Kernel)
  2. Close any other notebooks that might have the database open
  3. Then restart this notebook
- Alternatively, if the other notebook is already running, just use Cell 14+ in that notebook directly

## Cell 1: Imports & GPU Check

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient
import google.generativeai as genai
import os

# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA not available, using CPU")

## Cell 2: Load BGE Embedding Model

In [ ]:
# Load BGE embedding model
model_name = "BAAI/bge-large-en-v1.5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")
embedding_model = SentenceTransformer(model_name, device=device)

# Test embedding to get dimension
test_embedding = embedding_model.encode(["test"])
embedding_dim = len(test_embedding[0])

print(f"\nModel loaded successfully!")
print(f"Model: {model_name}")
print(f"Device: {device}")
print(f"Embedding dimension: {embedding_dim}")
print(f"Max sequence length: {embedding_model.max_seq_length}")

## Cell 3: Configuration

In [ ]:
# Configuration
MILVUS_DB = "milvus_all_docs_mixed.db"
COLLECTION_NAME = "rag_all_docs_mixed"
EMBEDDING_DIM = 1024

print("Configuration:")
print(f"  Milvus DB: {MILVUS_DB}")
print(f"  Collection name: {COLLECTION_NAME}")
print(f"  Expected embedding dimension: {EMBEDDING_DIM}")

## Cell 4: Connect to Existing Milvus Database

In [ ]:
# Initialize Milvus client (connects to existing DB)
print(f"Connecting to existing Milvus database: {MILVUS_DB}")

try:
    client = MilvusClient(MILVUS_DB)
    
    # Verify collection exists
    if client.has_collection(COLLECTION_NAME):
        print(f"✓ Collection '{COLLECTION_NAME}' found!")
        
        # Get basic stats
        stats = client.get_collection_stats(COLLECTION_NAME)
        print(f"\nCollection Statistics:")
        print(f"  Total vectors: {stats['row_count']}")
        
        if stats['row_count'] == 0:
            print(f"\n⚠️  WARNING: Collection exists but has ZERO vectors!")
            print(f"  This usually means the database file is locked by another notebook.")
            print(f"  Please shut down the kernel for 'phase2_chunking_embedding_mixed.ipynb' and retry.")
        else:
            print(f"  Collection loaded and ready for queries!")
    else:
        print(f"✗ ERROR: Collection '{COLLECTION_NAME}' not found!")
        print(f"\nPlease run 'phase2_chunking_embedding_mixed.ipynb' first to create the vector database.")
        raise ValueError(f"Collection '{COLLECTION_NAME}' does not exist in '{MILVUS_DB}'")
        
except Exception as e:
    if "file has been opened by another program" in str(e) or "Open local milvus failed" in str(e):
        print(f"\n✗ ERROR: Database is locked by another program!")
        print(f"\n📋 TROUBLESHOOTING STEPS:")
        print(f"  1. In Jupyter, go to Kernel → Shut Down Kernel for 'phase2_chunking_embedding_mixed.ipynb'")
        print(f"  2. Close any other notebooks that might have this database open")
        print(f"  3. Then restart this notebook")
        print(f"\n💡 TIP: If you already have the other notebook running, just use Cell 14+ in that notebook directly!")
        raise
    else:
        print(f"\n✗ ERROR: {e}")
        raise

## Cell 5: Collection Statistics & Summary

In [ ]:
# Get detailed collection statistics
stats = client.get_collection_stats(COLLECTION_NAME)

# Sample a few records to show metadata structure
print(f"\n{'='*80}")
print(f"COLLECTION DETAILS")
print(f"{'='*80}")
print(f"\nCollection Name: {COLLECTION_NAME}")
print(f"Total Vectors: {stats['row_count']}")
print(f"Embedding Dimension: {EMBEDDING_DIM}")

# Query a sample record to show metadata fields
print(f"\nSample Record (showing metadata fields):")
sample_results = client.search(
    collection_name=COLLECTION_NAME,
    data=[[0.0] * EMBEDDING_DIM],  # dummy vector
    limit=1,
    output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"]
)

if sample_results and len(sample_results[0]) > 0:
    sample = sample_results[0][0]['entity']
    print(f"  Source File: {sample['source_file']}")
    print(f"  Doc Type: {sample['doc_type']}")
    print(f"  Chunker Used: {sample['chunker_used']}")
    print(f"  Chunk ID: {sample['chunk_id'] + 1}/{sample['total_chunks']}")
    print(f"  Text Preview: {sample['text'][:150]}...")

print(f"\n{'='*80}")
print(f"Database ready for RAG queries!")
print(f"{'='*80}")

## Cell 6: Initialize Gemini 2.5 Flash for Answer Generation

In [ ]:
# Configure Gemini API
# Note: Set your GEMINI_API_KEY or GOOGLE_API_KEY environment variable
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY not found in environment variables")
    print("Please set GEMINI_API_KEY or GOOGLE_API_KEY to use Gemini for answer generation")
    print("Example: export GEMINI_API_KEY='your-api-key-here'")
else:
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-2.5-flash')
    
    print("Gemini 2.5 Flash initialized successfully!")
    print(f"Model: gemini-2.5-flash")
    print(f"Ready for RAG answer generation")

## Cell 7: RAG Function - Retrieve Context & Generate Answer

In [ ]:
def rag_query(query_text, top_k=5, show_full_chunks=True):
    """
    Complete RAG pipeline: Retrieve relevant chunks and generate answer using Gemini.
    
    Args:
        query_text: The question to answer
        top_k: Number of chunks to retrieve
        show_full_chunks: Whether to print full chunk text
    
    Returns:
        dict with answer, sources, and metadata
    """
    print(f"\n{'='*80}")
    print(f"QUERY: {query_text}")
    print(f"{'='*80}\n")
    
    # Step 1: Embed query
    print(f"Step 1: Embedding query...")
    query_embedding = embedding_model.encode([query_text])[0].tolist()
    
    # Step 2: Search vector database
    print(f"Step 2: Searching for top {top_k} relevant chunks...")
    search_results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"]
    )
    
    # Extract results
    context_texts = []
    sources = []
    
    print(f"\n{'='*80}")
    print(f"RETRIEVED CHUNKS:")
    print(f"{'='*80}\n")
    
    for idx, result in enumerate(search_results[0], 1):
        entity = result['entity']
        chunk_text = entity['text']
        context_texts.append(chunk_text)
        
        source_info = {
            'rank': idx,
            'source_file': entity['source_file'],
            'doc_type': entity['doc_type'],
            'chunk_id': entity['chunk_id'],
            'total_chunks': entity['total_chunks'],
            'chunker_used': entity['chunker_used'],
            'distance': result['distance'],
            'text': chunk_text
        }
        sources.append(source_info)
        
        # Display chunk info
        print(f"Chunk {idx}:")
        print(f"  Source: {entity['source_file']} ({entity['doc_type']})")
        print(f"  Chunker: {entity['chunker_used']}")
        print(f"  Chunk: {entity['chunk_id']+1}/{entity['total_chunks']}")
        print(f"  Distance: {result['distance']:.4f}")
        if show_full_chunks:
            print(f"  Text:\n{chunk_text}")
        else:
            print(f"  Text: {chunk_text[:200]}...")
        print(f"\n{'-'*80}\n")
    
    # Step 3: Build context for Gemini
    print(f"Step 3: Building context for LLM...")
    context = "\n\n".join([
        f"[Source {i+1}: {sources[i]['source_file']}]\n{text}" 
        for i, text in enumerate(context_texts)
    ])
    
    # Step 4: Generate answer with Gemini
    print(f"Step 4: Generating answer with Gemini 2.5 Flash...\n")
    
    prompt = f"""You are an expert financial analyst AI assistant with access to retrieved document context from a RAG (Retrieval-Augmented Generation) system.

IMPORTANT CONTEXT UNDERSTANDING:
The context provided below consists of relevant text chunks retrieved from financial documents (balance sheets, audit reports, financial statements, XBRL filings). These chunks may contain:
- Complete information directly answering the question
- Partial information requiring synthesis across multiple chunks
- Numerical data from tables requiring calculations
- Related information from which the answer can be derived through reasoning

YOUR CAPABILITIES:
1. **Multi-step reasoning**: You can derive answers through logical inference even if not explicitly stated
2. **Calculations**: You can perform arithmetic operations (addition, subtraction, multiplication, division, percentages, ratios, year-over-year changes)
3. **Cross-chunk synthesis**: Combine information from multiple retrieved chunks to form complete answers
4. **Financial domain knowledge**: Apply standard accounting principles and financial analysis techniques
5. **Pattern recognition**: Identify trends, anomalies, and relationships in financial data

RETRIEVED CONTEXT:
{context}

USER QUESTION: {query_text}

INSTRUCTIONS:
1. **Direct Information**: If the answer is explicitly stated in the context, provide it directly with source citations
2. **Derivable Information**: If the answer requires calculation or inference:
   - Clearly explain your reasoning steps
   - Show calculations with numbers from the context
   - Cite which sources provided the input data
3. **Incomplete Information**: If context provides partial information:
   - State what information is available
   - Explain what additional data would be needed
   - Provide the best possible answer with appropriate caveats
4. **No Information**: If context doesn't contain relevant information:
   - Explicitly state that the information is not available in the retrieved documents
   - Do NOT make up information or use knowledge outside the provided context

ANSWER FORMAT:
- Begin with a clear, concise answer
- Show calculations if performed (e.g., "Total Assets = 1,234,567 + 567,890 = 1,802,457")
- Cite sources: [Source 1], [Source 2], etc.
- If derived through reasoning, explain the logic
- Use proper financial terminology

Your response:"""
    
    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    # Display answer
    print(f"{'='*80}")
    print(f"GENERATED ANSWER:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n{'='*80}\n")
    
    return {
        'question': query_text,
        'answer': answer,
        'sources': sources,
        'num_chunks_retrieved': len(context_texts)
    }

print("RAG function ready!")
print("Usage: rag_query('your question here', top_k=5, show_full_chunks=True)")

## Cell 8: Test RAG - Example Query

In [ ]:
# Example query to test the RAG system
result = rag_query(
    "What is the Corporate Identity Number (CIN) of the company?",
    top_k=5,
    show_full_chunks=False  # Set to True to see full chunk text
)

## Cell 9: Custom Query Interface

Use this cell to run your own custom queries. You can:
- Ask single questions
- Process a list of questions
- Adjust `top_k` to retrieve more/fewer chunks
- Toggle `show_full_chunks` to control output verbosity

In [ ]:
# Single query example
custom_result = rag_query(
    "What is the name of the company?",
    top_k=5,
    show_full_chunks=True
)

In [ ]:
# Multiple queries example (uncomment to use)
"""
questions = [
    "What is the Corporate Identity Number (CIN) of the company?",
    "What is the name of the company?",
    "What is the address of the registered office?",
    "What is the email ID of the company?",
    "What is the financial year covered in the statements?",
    "What is the authorized capital of the company?",
    "Show me balance sheet information for ExampleCo",
    "What are the revenue figures for FY 23-24?"
]

results = []
for question in questions:
    result = rag_query(question, top_k=5, show_full_chunks=False)
    results.append(result)
    print("\n" + "="*80 + "\n")
"""
pass